In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

dokumen = [
    "Cinta Luar Biasa",
    "Hati-Hati di Jalan",
    "Sial",
    "Tak Ingin Usai",
    "Bertaut",
    "Runtuh",
    "Komang",
    "Pesan Terakhir",
    "Melukis Senja",
    "Satu Bulan"
]

queries = {
    "cinta hati": {1, 2},
    "jalan terakhir": {2, 8},
    "runtuh usai": {4, 6}
}

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(dokumen)


def precision_at_k(hasil, relevan, k):
    return sum(1 for d in hasil[:k] if d in relevan) / k


def recall(hasil, relevan):
    if not relevan:
        return 0.0
    return sum(1 for d in hasil if d in relevan) / len(relevan)


def f1(p, r):
    if p + r == 0:
        return 0.0
    return 2 * p * r / (p + r)


def average_precision(hasil, relevan):
    if not relevan:
        return 0.0

    hit = 0
    total = 0.0

    for i, d in enumerate(hasil, start=1):
        if d in relevan:
            hit += 1
            total += hit / i

    return total / len(relevan)


nilai_ap = []

for query, relevan in queries.items():

    print("\n" + "=" * 70)
    print("QUERY:", query)
    print("=" * 70)

    query_vector = vectorizer.transform([query])
    similarity = cosine_similarity(query_vector, tfidf_matrix).flatten()

    ranking = similarity.argsort()[::-1]
    hasil = [index + 1 for index in ranking]

    print("\n5 DOKUMEN TERATAS")

    for peringkat, index in enumerate(ranking[:5], start=1):
        print(f"\nPeringkat {peringkat}")
        print("ID Dokumen :", index + 1)
        print("Judul      :", dokumen[index])
        print("Skor       :", f"{similarity[index]:.4f}")

    p = precision_at_k(hasil, relevan, 5)
    r = recall(hasil, relevan)
    f = f1(p, r)
    ap = average_precision(hasil, relevan)

    nilai_ap.append(ap)

    print("\nGROUND TRUTH :", relevan)
    print("Precision@5  :", f"{p:.4f}")
    print("Recall       :", f"{r:.4f}")
    print("F1-Score     :", f"{f:.4f}")
    print("AP           :", f"{ap:.4f}")


MAP = sum(nilai_ap) / len(nilai_ap)

print("\n" + "=" * 70)
print("HASIL AKHIR")
print("=" * 70)
print("MAP :", f"{MAP:.4f}")